# 10 Results

No model is estimated here: the notebook reads the forecasts saved by notebooks 07–09 and writes the tables (LaTeX, `tables_ch4/`) and figures (PDF, `figures_ch4/`) of Chapter 4. No GPU needed.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Configuration

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "/content/drive/MyDrive/Thesis/data"
TAB_DIR = os.path.join(DATA_DIR, "tables_ch4")
FIG_DIR = os.path.join(DATA_DIR, "figures_ch4")
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SAMPLES = {"complete_case_mv25": "Complete case", "imputed_mv25": "Imputed"}
KEYS = ["symbol", "date"]
LINEAR_EXTRA = ["ridge", "lasso", "enet"]
NN = ["nn_1", "nn_2", "nn_3", "nn_4", "nn_5"]
MODELS = ["ols3", "ols_pars", "ols_full", "ridge", "lasso", "enet", "elanet",
          "rf", "gbrt"] + NN + ["comb"]
MODELS_DM = ["ols3", "ols_pars", "ols_full", "elanet", "rf", "gbrt"] + NN + ["comb"]
PERIODS = {"2010-2025": (2010, 2025), "2010-2020": (2010, 2020), "2021-2025": (2021, 2025)}
NW_LAGS = 4        # Newey-West lags, as in Section 3.7
N_DECILES = 10

# Drobetz and Otto (2021), Tables 2-3, p. 521: test period 2000-2020, R^2 in %
DO_R2 = {"ols_pars": 0.57, "ols_full": -0.52, "elanet": 1.21, "rf": 1.18,
         "gbrt": 1.14, "nn_1": 1.23}
DO_R2_SIZE = {"ols_pars": (0.11, 0.90), "ols_full": (-1.16, -0.10), "elanet": (0.75, 1.55),
              "rf": (0.92, 1.40), "gbrt": (0.74, 1.46), "nn_1": (0.93, 1.47)}   # (large, small)
DO_SLOPE = {"ols_pars": 0.65, "ols_full": 0.45, "elanet": 0.92, "rf": 1.03,
            "gbrt": 1.07, "nn_1": 0.98}

## Loading the forecasts

`ridge`, `lasso` and `enet` are only in `predictions_linear_*` and are merged on `symbol` and `date`. `comb` is fixed in advance: equal weight to each model family (elastic net, random forest, gradient boosting and the average of the five networks).

In [ ]:
def load_sample(tag):
    # Final forecasts of all models, plus ridge/lasso/enet from the linear file
    final = pd.read_parquet(os.path.join(DATA_DIR, f"predictions_final_{tag}.parquet"))
    lin = pd.read_parquet(os.path.join(DATA_DIR, f"predictions_linear_{tag}.parquet"),
                          columns=KEYS + LINEAR_EXTRA)
    df = final.merge(lin, on=KEYS, how="left", validate="one_to_one")
    missing = int(df[LINEAR_EXTRA].isna().sum().sum())
    assert missing == 0, f"{missing} linear forecasts not matched"
    # Forecast combination fixed ex ante: equal weight to each model family
    df["comb"] = (df["elanet"] + df["rf"] + df["gbrt"] + df[NN].mean(axis=1)) / 4
    df[MODELS] = df[MODELS].astype("float64")
    df["year"] = df["date"].dt.year
    return df

data = {tag: load_sample(tag) for tag in SAMPLES}
for tag, df in data.items():
    print(f"{SAMPLES[tag]:14s} {len(df):>8,} forecasts | {df['date'].nunique()} months | "
          f"{df['year'].min()}-{df['year'].max()}")

## Evaluation measures

As defined in Section 3.7.

In [ ]:
def in_period(df, period):
    a, b = PERIODS[period]
    return df[(df["year"] >= a) & (df["year"] <= b)]


def r2_by_year(df, models, target="target"):
    # Annual out-of-sample R^2 against a zero forecast (eq. 3.1), one column per model
    den = (df[target] ** 2).groupby(df["year"]).sum()
    return pd.DataFrame({m: 1 - ((df[target] - df[m]) ** 2).groupby(df["year"]).sum() / den
                         for m in models})


def r2_avg_annual(df, models, target="target"):
    return r2_by_year(df, models, target).mean()


def r2_pooled(df, models, target="target"):
    den = (df[target] ** 2).sum()
    return pd.Series({m: 1 - ((df[target] - df[m]) ** 2).sum() / den for m in models})


def demean_by_month(df, cols):
    # Subtract each month's cross-sectional mean (for the cross-sectional R^2, eq. 3.2)
    out = df[["date", "year"]].copy()
    for c in cols:
        out[c] = df[c] - df.groupby("date")[c].transform("mean")
    return out


def r2_cross_sectional(df, models, target="target"):
    return r2_pooled(demean_by_month(df, [target] + models), models, target)


def nw_tstat(x, lags=NW_LAGS):
    # Mean of x divided by its Newey-West standard error
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    T = len(x)
    u = x - x.mean()
    lrv = u @ u / T
    for l in range(1, lags + 1):
        lrv += 2 * (1 - l / (lags + 1)) * (u[l:] @ u[:-l]) / T
    return x.mean() / np.sqrt(lrv / T)


def dm_matrix(df, models, target="target"):
    # Modified Diebold-Mariano statistics (eq. 3.3): positive = column model more accurate
    e2 = pd.DataFrame({m: (df[target] - df[m]) ** 2 for m in models}).groupby(df["date"]).mean()
    out = pd.DataFrame(np.nan, index=models, columns=models)
    for i, a in enumerate(models):
        for b in models[i + 1:]:
            out.loc[a, b] = nw_tstat(e2[a] - e2[b])
            out.loc[b, a] = -out.loc[a, b]
    return out


def predictive_slope(df, models, target="target", rel_tol=0.1):
    # Average over test years of Cov(r, f)/Var(f). Years in which a model's forecasts are
    # nearly constant (sd below 10% of its median yearly sd) make the ratio unstable and
    # are excluded from the average and counted separately.
    rows = {}
    y = df[target]
    for m in models:
        f = df[m]
        g = df["year"]
        mf, my = f.groupby(g).mean(), y.groupby(g).mean()
        cov = (y * f).groupby(g).mean() - my * mf
        var = (f ** 2).groupby(g).mean() - mf ** 2
        slope = cov / var
        sd = np.sqrt(var)
        unstable = sd < rel_tol * sd.median()
        rows[m] = {"slope": slope[~unstable].mean(), "unstable_years": int(unstable.sum()),
                   "slope_all_years": slope.mean()}
    return pd.DataFrame(rows).T


def size_split(df):
    # Large and small firms, split each month at the median market capitalisation
    med = df.groupby("date")["MV"].transform("median")
    return df[df["MV"] >= med], df[df["MV"] < med]


def decile_portfolios(df, model, n=N_DECILES, ret="raw_target"):
    # Monthly sorts on the forecast; top and bottom deciles and long-short, EW and VW
    d = df[["date", ret, "MV", model]].dropna()
    rank = d.groupby("date")[model].rank(method="first")
    # Equal-sized deciles on the within-month rank (as in notebooks 07-09); ties broken by row order
    dec = rank.groupby(d["date"]).transform(lambda x: pd.qcut(x, n, labels=False)) + 1
    res = {}
    for name, sel in (("top", dec == n), ("bottom", dec == 1)):
        s = d[sel]
        res[(name, "EW")] = s.groupby("date")[ret].mean()
        res[(name, "VW")] = (s[ret] * s["MV"]).groupby(s["date"]).sum() / s.groupby("date")["MV"].sum()
    out = pd.DataFrame(res)
    for w in ("EW", "VW"):
        out[("ls", w)] = out[("top", w)] - out[("bottom", w)]
    return out


def perf(x):
    # Mean monthly return (%), Newey-West t-statistic, annualised Sharpe ratio
    x = x.dropna()
    return pd.Series({"Mean": 100 * x.mean(), "t": nw_tstat(x),
                      "SR": np.sqrt(12) * x.mean() / x.std()})

### Consistency check

Selected figures recomputed here against those printed by notebooks 07–09.

In [ ]:
cc, imp = data["complete_case_mv25"], data["imputed_mv25"]
full = "2010-2025"
checks = [
    ("CC avg annual R2 elanet",  100 * r2_avg_annual(cc, ["elanet"])["elanet"],          0.790),
    ("CC avg annual R2 rf",      100 * r2_avg_annual(cc, ["rf"])["rf"],                  0.821),
    ("CC avg annual R2 nn_5",    100 * r2_avg_annual(cc, ["nn_5"])["nn_5"],              0.700),
    ("Imp avg annual R2 rf",     100 * r2_avg_annual(imp, ["rf"])["rf"],                 0.805),
    ("CC pooled R2 rf",          100 * r2_pooled(cc, ["rf"])["rf"],                      0.652),
    ("CC CS R2 rf (pooled)",     100 * r2_cross_sectional(cc, ["rf"])["rf"],             0.293),
    ("Imp CS R2 rf (pooled)",    100 * r2_cross_sectional(imp, ["rf"])["rf"],            0.734),
    ("CC corr(R2, market) rf",   r2_by_year(cc, ["rf"])["rf"].corr(cc.groupby("year")["target"].mean()), 0.79),
    ("CC slope rf (all years)",  predictive_slope(cc, ["rf"]).loc["rf", "slope_all_years"], 1.162),
    ("CC slope rf (excl.)",      predictive_slope(cc, ["rf"]).loc["rf", "slope"],        1.162),
    ("Imp slope gbrt (all years)", predictive_slope(imp, ["gbrt"]).loc["gbrt", "slope_all_years"], 0.995),
    ("CC DM ols_pars -> rf",     dm_matrix(cc, ["ols_pars", "rf"]).loc["ols_pars", "rf"], 3.75),
    ("Imp DM ols3 -> rf",        dm_matrix(imp, ["ols3", "rf"]).loc["ols3", "rf"],       3.82),
    ("CC large R2 rf",           100 * r2_avg_annual(size_split(cc)[0], ["rf"])["rf"],   0.925),
    ("CC small R2 rf",           100 * r2_avg_annual(size_split(cc)[1], ["rf"])["rf"],   0.773),
    ("CC VW long-short rf (%)",  perf(decile_portfolios(cc, "rf")[("ls", "VW")])["Mean"], 1.05),
    ("CC VW long-short rf (t)",  perf(decile_portfolios(cc, "rf")[("ls", "VW")])["t"],   2.24),
    ("Imp VW long-short gbrt (%)", perf(decile_portfolios(imp, "gbrt")[("ls", "VW")])["Mean"], 1.96),
    ("Imp VW long-short gbrt (t)", perf(decile_portfolios(imp, "gbrt")[("ls", "VW")])["t"], 3.55),
]
print(pd.DataFrame(checks, columns=["quantity", "this notebook", "notebooks 07-09"])
        .set_index("quantity").round(3).to_string())

## LaTeX tables

`booktabs` tables, included in the thesis with `\input{}`. The notes under each table are drafts; the final notes are in the thesis.

In [ ]:
def _cell(v, digits):
    if pd.isna(v):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{v:,}"
    s = f"{v:.{digits}f}"
    return f"${s}$" if v < 0 else s


def _tex(s):
    # Escape text for LaTeX; model names in typewriter font; year ranges with an en dash
    s = str(s)
    if s == "t":
        return "$t$"
    if s in MODELS or s == "nn":
        return r"\texttt{" + s.replace("_", r"\_") + "}"
    s = re.sub(r"(\d{4})-(\d{4})", r"\1--\2", s)
    s = s.replace("_", r"\_").replace("%", r"\%")
    return re.sub(r"\bR2\b", r"$R^2$", s)


def latex_table(df, fname, caption, label, digits=2, notes=None, size=r"\small"):
    # df: rows = models (or labels), columns = flat or two-level MultiIndex
    two_level = isinstance(df.columns, pd.MultiIndex)
    ncol = df.shape[1]
    lines = [r"\begin{table}[htbp]", r"\centering", size,
             rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{@{}l" + "r" * ncol + r"@{}}", r"\toprule"]
    if two_level:
        top, bottom = df.columns.get_level_values(0), df.columns.get_level_values(1)
        groups, start = [], 0
        for i in range(1, ncol + 1):
            if i == ncol or top[i] != top[start]:
                groups.append((top[start], start, i))
                start = i
        lines.append(" & " + " & ".join(rf"\multicolumn{{{e - s}}}{{c}}{{{_tex(g)}}}"
                                        for g, s, e in groups) + r" \\")
        lines.append("".join(rf"\cmidrule(lr){{{s + 2}-{e + 1}}}" for g, s, e in groups))
        lines.append(" & " + " & ".join(_tex(b) for b in bottom) + r" \\")
    else:
        lines.append(" & " + " & ".join(_tex(c) for c in df.columns) + r" \\")
    lines.append(r"\midrule")
    for idx, row in df.astype(object).iterrows():    # object keeps integer columns as integers
        lines.append(_tex(idx) + " & " + " & ".join(_cell(v, digits) for v in row) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    if notes:
        lines += [r"\\[0.5em]", r"\begin{minipage}{0.95\textwidth}",
                  r"\scriptsize \textit{Notes.} " + notes, r"\end{minipage}"]
    lines.append(r"\end{table}")
    path = os.path.join(TAB_DIR, fname)
    with open(path, "w") as fh:
        fh.write("\n".join(lines) + "\n")
    print("saved", path)

## Out-of-sample R² (Section 4.1)



In [ ]:
r2_avg, r2_pool = {}, {}
for tag, s in SAMPLES.items():
    for p in PERIODS:
        sub = in_period(data[tag], p)
        r2_avg[(s, p)] = 100 * r2_avg_annual(sub, MODELS)
        r2_pool[(s, p)] = 100 * r2_pooled(sub, MODELS)
r2_avg, r2_pool = pd.DataFrame(r2_avg), pd.DataFrame(r2_pool)
r2_avg[("Drobetz--Otto", "2000-2020")] = pd.Series(DO_R2)

print("Average annual R^2 (%)")
display(r2_avg.round(2))
print("Pooled R^2 (%)")
display(r2_pool.round(2))

latex_table(r2_avg, "tab_r2_avg_annual.tex",
            "Average annual out-of-sample $R^2$ (\\%)", "tab:r2_avg",
            notes="Out-of-sample $R^2$ against a zero forecast (equation 3.1), computed in each test "
                  "year and averaged across years. Values for Drobetz and Otto (2021) from their Tables 2--3.")
latex_table(r2_pool, "tab_r2_pooled.tex",
            "Pooled out-of-sample $R^2$ (\\%)", "tab:r2_pooled",
            notes="Out-of-sample $R^2$ against a zero forecast (equation 3.1), computed on all test "
                  "observations of each period at once.")

In [ ]:
# Annual R^2 by test year (used for Figure 4.1 and for the discussion of individual years)
r2_year = {SAMPLES[tag]: 100 * r2_by_year(df, MODELS) for tag, df in data.items()}
for s, t in r2_year.items():
    print(s)
    display(t[["ols3", "ols_pars", "elanet", "rf", "gbrt", "nn_3", "nn_5", "comb"]].round(2))
    t.round(4).to_csv(os.path.join(TAB_DIR, f"r2_by_year_{s.replace(' ', '_').lower()}.csv"))

## Cross-sectional R² and the market (Section 4.1)

In [ ]:
cs = {}
for tag, s in SAMPLES.items():
    for p in PERIODS:
        cs[(s, p)] = 100 * r2_cross_sectional(in_period(data[tag], p), MODELS)
cs = pd.DataFrame(cs)

corr = {}
for tag, s in SAMPLES.items():
    df = data[tag]
    corr[s] = r2_by_year(df, MODELS).corrwith(df.groupby("year")["target"].mean())
corr = pd.DataFrame(corr)

print("Cross-sectional R^2 (%)")
display(cs.round(2))
print("Correlation of annual R^2 with the year's average excess return")
display(corr.round(2))

latex_table(cs, "tab_r2_cross_sectional.tex",
            "Cross-sectional out-of-sample $R^2$ (\\%)", "tab:r2_cs",
            notes="Equation 3.2: returns and forecasts are demeaned by their cross-sectional average "
                  "in each month before computing the $R^2$.")
latex_table(corr, "tab_r2_market_corr.tex",
            "Correlation between annual $R^2$ and the average excess return of the year", "tab:r2_corr")

## Diebold–Mariano tests (Section 4.2)

Positive: the column model is more accurate than the row model. As in Gu, Kelly and Xiu (2020, Table 3), the LaTeX table shows only the upper triangle.

In [ ]:
dm = {}
for tag, s in SAMPLES.items():
    dm[s] = dm_matrix(data[tag], MODELS_DM)
    print(s)
    display(dm[s].round(2))
    upper = dm[s].where(np.triu(np.ones(dm[s].shape, dtype=bool), k=1))
    upper = upper.iloc[:-1, 1:]           # drop the empty last row and first column
    latex_table(upper, f"tab_dm_{tag}.tex",
                f"Diebold--Mariano tests, {s.lower()} sample, 2010--2025", f"tab:dm_{tag}",
                size=r"\footnotesize",
                notes="Modified Diebold--Mariano statistic (equation 3.3) with Newey--West standard "
                      "errors (four lags). Positive values indicate that the column model is more accurate.")

## Large and small firms (Section 4.3), predictive slopes (Section 4.2)

In [ ]:
size_tab = {}
for tag, s in SAMPLES.items():
    large, small = size_split(data[tag])
    size_tab[(s, "Large")] = 100 * r2_avg_annual(large, MODELS)
    size_tab[(s, "Small")] = 100 * r2_avg_annual(small, MODELS)
size_tab = pd.DataFrame(size_tab)
size_tab[("Drobetz--Otto", "Large")] = pd.Series({m: v[0] for m, v in DO_R2_SIZE.items()})
size_tab[("Drobetz--Otto", "Small")] = pd.Series({m: v[1] for m, v in DO_R2_SIZE.items()})
print("Average annual R^2 (%) by size, 2010-2025")
display(size_tab.round(2))
latex_table(size_tab, "tab_r2_size.tex",
            "Average annual out-of-sample $R^2$ (\\%) for large and small firms, 2010--2025", "tab:r2_size",
            notes="Firms are split each month at the median market capitalisation of the sample. "
                  "Values for Drobetz and Otto (2021) refer to 2000--2020.")

slopes = {}
for tag, s in SAMPLES.items():
    sl = predictive_slope(data[tag], MODELS)
    slopes[(s, "Slope")] = sl["slope"]
    slopes[(s, "Excl. years")] = sl["unstable_years"].astype(int)
    print(s, "- slope averaged over all years, for comparison:")
    print(sl["slope_all_years"].round(3).to_string())
slopes = pd.DataFrame(slopes)
slopes[("Drobetz--Otto", "Slope")] = pd.Series(DO_SLOPE)
display(slopes.round(3))
latex_table(slopes, "tab_slopes.tex", "Predictive slopes, 2010--2025", "tab:slopes",
            notes="Slope of a regression of realised on forecast returns in each test year, averaged "
                  "across years. Years in which a model's forecasts are nearly constant are excluded "
                  "and counted in the second column.")

## Long-short portfolios (Section 4.4)

In [ ]:
ls_series = {}
port_tabs = {}
for tag, s in SAMPLES.items():
    rows = {}
    for m in MODELS:
        p = decile_portfolios(data[tag], m)
        ls_series[(s, m)] = p[[("ls", "EW"), ("ls", "VW")]]
        rows[m] = pd.concat({"Equal-weighted": perf(p[("ls", "EW")]),
                             "Value-weighted": perf(p[("ls", "VW")])})
    port_tabs[s] = pd.DataFrame(rows).T
    print(s, "- long-short decile portfolios (mean in % per month)")
    display(port_tabs[s].round(2))
    latex_table(port_tabs[s], f"tab_portfolios_{tag}.tex",
                f"Long-short decile portfolios, {s.lower()} sample, 2010--2025", f"tab:port_{tag}",
                notes="Each month stocks are sorted into deciles on their forecast; the portfolio buys "
                      "the top decile and sells the bottom one. Mean monthly excess return (\\%), "
                      "Newey--West $t$-statistic and annualised Sharpe ratio, on non-winsorised returns.")

## Comparing the samples (Section 4.3)

Same observations: models estimated on the two samples, evaluated on the complete-case observations and target; positive DM: the model estimated on the imputed sample is more accurate. Recovered observations: those present only in the imputed sample, against those retained.

In [ ]:
m_cmp = [m for m in MODELS if m != "comb"] + ["comb"]
common = cc.merge(imp[KEYS + m_cmp], on=KEYS, suffixes=("", "_imp"))
print(f"Common observations: {len(common):,} of {len(cc):,} complete-case forecasts")

same = {}
for m in m_cmp:
    same[m] = {
        "R2 CC-trained": 100 * r2_avg_annual(common, [m])[m],
        "R2 imputed-trained": 100 * r2_avg_annual(common.assign(**{m: common[m + "_imp"]}), [m])[m],
        "DM": nw_tstat(((common["target"] - common[m]) ** 2).groupby(common["date"]).mean()
                       - ((common["target"] - common[m + "_imp"]) ** 2).groupby(common["date"]).mean()),
    }
same = pd.DataFrame(same).T

in_cc = imp.set_index(KEYS).index.isin(cc.set_index(KEYS).index)
retained, recovered = imp[in_cc], imp[~in_cc]
rec = pd.DataFrame({
    "R2 recovered": 100 * r2_avg_annual(recovered, m_cmp),
    "R2 retained": 100 * r2_avg_annual(retained, m_cmp),
    "CS R2 recovered": 100 * r2_cross_sectional(recovered, m_cmp),
    "CS R2 retained": 100 * r2_cross_sectional(retained, m_cmp),
})
print(f"Recovered: {len(recovered):,} forecasts ({len(recovered) / len(imp):.1%}); median market value "
      f"{recovered['MV'].median():.1f} mn against {retained['MV'].median():.1f} mn retained")
display(same.round(2))
display(rec.round(2))

latex_table(same, "tab_same_observations.tex",
            "Models estimated on the two samples, evaluated on the same complete-case observations, 2010--2025",
            "tab:same_obs",
            notes="Average annual out-of-sample $R^2$ (\\%). DM: Diebold--Mariano statistic, positive when "
                  "the model estimated on the imputed sample is more accurate.")
latex_table(rec, "tab_recovered.tex",
            "Recovered and retained observations, models estimated on the imputed sample, 2010--2025",
            "tab:recovered",
            notes="Recovered: observations present only in the imputed sample. Retained: observations "
                  "also present in the complete-case sample. $R^2$ in \\%.")

## Figures

In [ ]:
# One model per family; nn_3 is the network Gu, Kelly and Xiu single out as their best
FIG_MODELS = ["elanet", "rf", "gbrt", "nn_3"]
COLORS = {"elanet": "#2a78d6", "rf": "#eb6834", "gbrt": "#1baf7a", "nn_3": "#eda100", "comb": "#e87ba4"}
BENCH = {"ols_pars": "#8a8984"}
INK, GRID = "#52514e", "#e5e4df"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK, "axes.labelcolor": INK,
                     "xtick.color": INK, "ytick.color": INK, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})


def style(ax):
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)


# Figure 4.1: annual R^2 by test year, with the year's average excess return below
fig, axes = plt.subplots(2, 2, figsize=(9, 5.2), sharex=True,
                         gridspec_kw={"height_ratios": [2.2, 1]})
for j, (tag, s) in enumerate(SAMPLES.items()):
    t = r2_year[s]
    ax = axes[0, j]
    for m, c in BENCH.items():
        ax.plot(t.index, t[m], color=c, lw=1.4, ls="--", label=m)
    for m in FIG_MODELS:
        ax.plot(t.index, t[m], color=COLORS[m], lw=1.6, marker="o", ms=3, label=m)
    ax.axhline(0, color=INK, lw=0.6)
    ax.set_title(s, fontsize=10, color=INK)
    ax.set_ylabel("Out-of-sample $R^2$ (%)" if j == 0 else "")
    style(ax)
    mkt = 100 * data[tag].groupby("year")["target"].mean()
    ax2 = axes[1, j]
    ax2.bar(mkt.index, mkt.values, color="#b9b8b2", width=0.7)
    ax2.axhline(0, color=INK, lw=0.6)
    ax2.set_ylabel("Avg. excess\nreturn (%)" if j == 0 else "")
    style(ax2)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(labels), loc="upper center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(os.path.join(FIG_DIR, "fig_r2_by_year.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Figure 4.2: cumulative value of 1 euro in the value-weighted long-short portfolios
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for j, (tag, s) in enumerate(SAMPLES.items()):
    ax = axes[j]
    for m in FIG_MODELS + ["comb"]:
        r = ls_series[(s, m)][("ls", "VW")]
        ax.plot(r.index, (1 + r).cumprod(), color=COLORS[m], lw=1.6, label=m)
    ax.axhline(1, color=INK, lw=0.6)
    ax.set_yscale("log")                  # log scale: equal slopes = equal monthly returns
    ax.set_title(s, fontsize=10, color=INK)
    ax.set_ylabel("Value of €1 (log scale)" if j == 0 else "")
    style(ax)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(labels), loc="upper center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(os.path.join(FIG_DIR, "fig_ls_cumulative_vw.pdf"), bbox_inches="tight")
plt.show()

## Extreme months and concentration (Section 4.4)

Months with the largest value-weighted long-short returns and the stocks behind them, the weight of the largest stock in each leg, and Bankia in May 2013.

In [ ]:
# Months with extreme value-weighted long-short returns and the stocks behind them
def ls_contributions(df, model, month, n=N_DECILES, ret="raw_target"):
    # Same decile rule as decile_portfolios; contribution = side x weight x return
    d = df[df["date"] == month][["symbol", "country", "MV", "target", ret, model]].copy()
    d["decile"] = pd.qcut(d[model].rank(method="first"), n, labels=False) + 1
    d = d[d["decile"].isin([1, n])].copy()
    d["side"] = np.where(d["decile"] == n, "long", "short")
    d["weight"] = d["MV"] / d.groupby("side")["MV"].transform("sum")
    d["contrib_%"] = 100 * np.where(d["side"] == "long", 1, -1) * d["weight"] * d[ret]
    return d.sort_values("contrib_%", key=np.abs, ascending=False)


DIAG_MODELS = ["elanet", "rf", "gbrt", "comb"]
for tag, s in SAMPLES.items():
    ls = pd.DataFrame({m: 100 * ls_series[(s, m)][("ls", "VW")] for m in DIAG_MODELS})
    extreme = ls.abs().max(axis=1).nlargest(5).index
    print(f"\n=== {s}: 5 months with the most extreme VW long-short return (%), formation month ===")
    print(ls.loc[extreme].round(1).to_string())
    print("Mean VW long-short (%) with and without these 5 months:")
    print(pd.DataFrame({"all months": ls.mean(), "without": ls.drop(extreme).mean()}).round(2).T.to_string())
    for month in extreme[:3]:
        m = ls.loc[month].abs().idxmax()
        c = ls_contributions(data[tag], m, month)
        print(f"\n{s} | {m} | formation {month:%Y-%m} | long-short {ls.loc[month, m]:.1f}% | top 5 stocks:")
        print(c.head(5)[["symbol", "country", "side", "MV", "weight", "raw_target", "target", "contrib_%"]]
              .round(3).to_string(index=False))


# Concentration: average weight of the largest stock in each leg of the VW portfolio
def max_leg_weight(df, model, n=N_DECILES):
    d = df[["date", "MV", model]].copy()
    d["decile"] = d.groupby("date")[model].rank(method="first").groupby(d["date"]).transform(
        lambda x: pd.qcut(x, n, labels=False)) + 1
    d = d[d["decile"].isin([1, n])]
    w = d["MV"] / d.groupby(["date", "decile"])["MV"].transform("sum")
    mx = w.groupby([d["date"], d["decile"]]).max().unstack()
    return pd.Series({"long": mx[n].mean(), "short": mx[1].mean(),
                      "stocks per leg": d.groupby(["date", "decile"]).size().mean()})

print("\nAverage weight of the largest stock in each VW leg")
for tag, s in SAMPLES.items():
    print(s)
    print(pd.DataFrame({m: max_leg_weight(data[tag], m) for m in DIAG_MODELS}).round(2).to_string())

In [ ]:
# Bankia: market value around the May 2013 recapitalisation
print(imp[imp["symbol"] == "E:BKIA"].set_index("date").loc["2012-12-01":"2013-08-01", ["MV", "raw_target"]])

# Imputed sample: value-weighted long-short without May 2013
for m in ["rf", "gbrt", "comb"]:
    r = ls_series[("Imputed", m)][("ls", "VW")]
    print(m, perf(r.drop(pd.Timestamp("2013-05-01"))).round(2).to_dict())